# Docker Image for an AI App

**Phase 06** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-06/05-docker-image-ai-app.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.

> **Note:** This lesson builds a service that must run as a long-lived process. The cells below show the full implementation but cannot run end-to-end in Colab. To run this lesson: clone the repo locally and use `uvicorn main:app` or `docker compose up`.


In [ ]:
!pip install -q anthropic fastapi pydantic uvicorn

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You have a working FastAPI AI service on your laptop. A colleague clones the repo and gets a different Python version, a missing system library, and an import error you have never seen. Your staging environment runs fine; production throws a segfault because the base image has a different glibc. The on-call engineer cannot reproduce the bug because they are on macOS and the server is Linux.

This is not a Python problem. It is a packaging problem. Every AI service that reaches production must answer one question with certainty: "Given this exact image, on any machine that can run Docker, does the service start and respond correctly?" If the answer is anything other than "yes," you have not s...

## The Concept

### Docker Layers and the Build Cache

A Docker image is a stack of read-only layers. Each instruction in a Dockerfile (`FROM`, `RUN`, `COPY`, `ENV`) creates one layer. Docker caches every layer by its instruction and its inputs. When a layer's inputs change, Docker invalidates that layer and every layer after it.

```
┌─────────────────────────────────────────────────────────────┐
│  FROM python:3.12-slim                  layer 1 (base OS)   │
├─────────────────────────────────────────────────────────────┤
│  RUN apt-get install ...                layer 2 (sys deps)  │
├──────────────────────...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Production FastAPI AI service for Docker packaging.

Reads config from environment variables (non-secret values only).
Expects ANTHROPIC_API_KEY to be injected at runtime via docker run --env.

Usage:
    docker build -t ai-app:latest -f Dockerfile .
    docker run -p 8000:8000 --env ANTHROPIC_API_KEY=sk-... ai-app:latest

    # Local dev (without Docker):
    ANTHROPIC_API_KEY=sk-... uvicorn main:app --reload
"""

import os

import anthropic
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="AI App", version="1.0")

# API key is required; fail fast at startup if missing
_api_key = os.environ.get("ANTHROPIC_API_KEY")
if not _api_key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY environment variable is required. "
        "Inject it at runtime: docker run --env ANTHROPIC_API_KEY=sk-..."
    )

client = anthropic.Anthropic(api_key=_api_key)

# Non-secret config from environment variables (set in Dockerfile or overridden at runtime)
MODEL = os.environ.get("MODEL", "claude-3-5-haiku-20241022")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "1024"))

### Request / Response models

In [ ]:
class GenerateRequest(BaseModel):
    prompt: str


class GenerateResponse(BaseModel):
    text: str
    model: str
    input_tokens: int
    output_tokens: int


class HealthResponse(BaseModel):
    status: str
    model: str

### Routes

In [ ]:
@app.get("/health", response_model=HealthResponse)
def health():
    """
    Health check endpoint used by Docker HEALTHCHECK and load balancers.
    Returns 200 when the service is ready to handle requests.
    Does NOT make an API call -- health checks must be cheap and fast.
    """
    return HealthResponse(status="ok", model=MODEL)


@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    """
    Send a prompt to Claude and return the generated text with token counts.

    Example:
        curl -X POST http://localhost:8000/generate \\
          -H "Content-Type: application/json" \\
          -d '{"prompt": "Explain Docker layers in one sentence."}'
    """
    if not req.prompt.strip():
        raise HTTPException(status_code=400, detail="prompt must not be empty")

    msg = client.messages.create(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": req.prompt}],
    )

    return GenerateResponse(
        text=msg.content[0].text,
        model=msg.model,
        input_tokens=msg.usage.input_tokens,
        output_tokens=msg.usage.output_tokens,
    )


# ---------------------------------------------------------------------------
# Dev entrypoint
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import uvicorn

port = int(os.environ.get("PORT", "8000"))
uvicorn.run("main:app", host="0.0.0.0", port=port, reload=True)

---

## Self-Check

Answer these without running code first:

**1. What is the performance problem with this layer order, and which layer is incorrectly positioned?**

- A. No problem. Docker is smart enough to detect that only code changed and will skip the pip install.
- B. The COPY . . layer invalidates the cache for the RUN pip install layer, forcing a full reinstall on every code change.
- C. The problem is in the FROM instruction. Using python:3.12 instead of python:3.12-slim causes slow builds.
- D. The RUN apt-get install layer must come after pip install to avoid dependency conflicts.

**2. What is the concrete security risk of this approach?**

- A. No risk. ENV variables in a Dockerfile are encrypted at rest in the image.
- B. The API key will appear in docker history output and in the image manifest, visible to anyone who can pull the image.
- C. The risk is that the key will expire inside the container after 24 hours due to Docker's security policy.
- D. ENV variables are not accessible inside the container at runtime, so the service will fail to start.

**3. What is the primary production benefit of the multi-stage build pattern for an AI service?**

- A. Two FROM statements allow you to use two different base OS distributions in the same container at runtime.
- B. The build stage installs compilers and dev tools to build wheels; the runtime stage copies only the compiled packages, producing a smaller image with fewer installed tools (smaller attack surface).
- C. Multi-stage builds are required by Kubernetes and will not deploy without them.
- D. The second FROM statement resets the working directory, which is required for uvicorn to find main.py.

**4. What is the most likely root cause, and what change would fix it?**

- A. The container is out of memory. Increase the task memory allocation.
- B. The Dockerfile has no HEALTHCHECK instruction, so ECS cannot determine if the service is ready and assumes it failed.
- C. The /health endpoint is returning 200 too quickly. ECS expects a delay of at least 5 seconds before the first healthy response.
- D. ECS requires the CMD to use exec form ["uvicorn", ...] not shell form. Shell form causes the health check to fail.

**5. Which two Dockerfile instructions together implement the non-root user requirement?**

- A. EXPOSE 8000 and CMD ["--no-root"] passed to uvicorn
- B. RUN useradd -m -u 1000 appuser followed by USER appuser before the CMD instruction
- C. RUN chmod 755 /app followed by ENV USER=appuser
- D. FROM python:3.12-slim automatically runs as non-root; no additional instruction is needed

**6. What is the cause of this problem and how do you fix it?**

- A. The Docker daemon is running out of disk space. Delete old images with docker system prune.
- B. There is no .dockerignore file, so Docker sends the entire directory including .git and .venv to the build daemon. Add a .dockerignore that excludes these directories.
- C. The FROM instruction must appear first in the Dockerfile for Docker to optimize the build context.
- D. The Dockerfile must use ADD instead of COPY to trigger build context compression.

**7. What is the correct way to handle these three values across the Dockerfile and docker run command?**

- A. Set all three in ENV instructions in the Dockerfile so the image is self-contained and no flags are needed at docker run.
- B. Set LOG_LEVEL and MODEL in ENV in the Dockerfile. Pass ANTHROPIC_API_KEY at runtime with docker run --env ANTHROPIC_API_KEY=$KEY.
- C. Pass all three at runtime with docker run --env. Never put any value in the Dockerfile.
- D. Store all three in a .env file and use COPY .env . in the Dockerfile so the container can load them automatically.

**8. Which combination of commands gives you the most direct evidence for all three requirements?**

- A. docker logs, docker stats, docker top
- B. docker history (for secrets), docker exec whoami (for user), docker inspect --format='{{.State.Health.Status}}' (for health after running)
- C. docker scan (for all three requirements at once)
- D. docker manifest inspect (for secrets and user) and docker run --health-check (for health)

_Answers are in `checks.json` in the lesson directory._